# Marine Corps sample
One race, under 20k rows. Run after `features.ipynb`.

In [1]:
import pandas as pd

df = pd.read_csv("data/marathons_features.csv", low_memory=False)
mcm = df[df["Marathon"] == "MarineCorps"]
mcm.groupby("Year").agg(runners=("Slow", "size"), slow_share=("Slow", "mean"),
                        has_history=("prev_races", lambda s: (s > 0).mean())).round(3)

,runners,slow_share,has_history
Year,,,
2000,17001,0.087,0.012
2001,14206,0.061,0.243
2002,14076,0.071,0.323
2003,15948,0.085,0.363
2004,16292,0.115,0.382
2005,19100,0.107,0.404
2006,20856,0.080,0.401
2007,20622,0.077,0.408
2008,18219,0.072,0.422


## Years
- Last 10 years: the field changed (older, more women, slower) → recent years are more representative.
- 2019 dropped: no ages → weak linking.
- Train 2009–2015, test 2016–2018. History still uses all races since 2000.

In [2]:
mcm = mcm[mcm["Year"].between(2009, 2018)]
mcm.groupby("Year").agg(median_time=("Time", "median"), slow_share=("Slow", "mean"),
                        female=("Sex", lambda s: (s == "F").mean()), median_age=("Age", "median")).round(2)

,median_time,slow_share,female,median_age
Year,,,,
2009,276.82,0.08,0.40,37.0
2010,280.98,0.09,0.40,37.0
2011,279.25,0.08,0.41,38.0
2012,282.30,0.08,0.42,38.0
2013,279.60,0.09,0.42,38.0
2014,288.45,0.11,0.44,39.0
2015,290.65,0.13,0.45,40.0
2016,298.93,0.17,0.46,40.0
2017,300.78,0.17,0.46,40.0


## Sample per year
Natural slow share kept (no oversampling).

In [3]:
TRAIN_PER_YEAR, TEST_PER_YEAR = 1700, 2000

parts = []
for year, g in mcm.groupby("Year"):
    n = TEST_PER_YEAR if year >= 2016 else TRAIN_PER_YEAR
    parts.append(g.sample(n=n, random_state=42))
sample = pd.concat(parts)
len(sample)

17900

## Columns
Drop constants (`Marathon`, `Month`) and the leaky `time_vs_prev_median`.

In [4]:
sample = sample.drop(columns=["Marathon", "Month", "time_vs_prev_median"]).reset_index(drop=True)
sample.columns.tolist()

['Sex',
 'Age',
 'Agegroup',
 'Time',
 'Year',
 'Slow',
 'linked_by_agegroup',
 'linkable',
 'course_difficulty',
 'course_slow_rate',
 'course_prev_size',
 'prev_races',
 'prev_median_time',
 'prev_best_time',
 'prev_slow_rate',
 'prev_median_rel_time',
 'prev_median_adj_time',
 'prev_best_adj_time',
 'prev_median_course_difficulty',
 'prev_distinct_races',
 'days_since_last',
 'prev_agegroup_links',
 'prev_last_time',
 'prev_same_races',
 'prev_same_median_time',
 'first_marathon',
 'expected_time']

## Check

In [5]:
sample.groupby(sample["Year"] >= 2016).agg(rows=("Slow", "size"), slow_share=("Slow", "mean"),
                                            has_history=("prev_races", lambda s: (s > 0).mean())).round(3)

,rows,slow_share,has_history
Year,,,
False,11900,0.093,0.453
True,6000,0.156,0.534


In [6]:
sample.isna().mean().round(3)

Sex                              0.000
Age                              0.000
Agegroup                         0.000
Time                             0.000
Year                             0.000
Slow                             0.000
linked_by_agegroup               0.000
linkable                         0.000
course_difficulty                0.000
course_slow_rate                 0.000
course_prev_size                 0.000
prev_races                       0.000
prev_median_time                 0.520
prev_best_time                   0.520
prev_slow_rate                   0.520
prev_median_rel_time             0.520
prev_median_adj_time             0.520
prev_best_adj_time               0.520
prev_median_course_difficulty    0.520
prev_distinct_races              0.000
days_since_last                  0.520
prev_agegroup_links              0.000
prev_last_time                   0.520
prev_same_races                  0.000
prev_same_median_time            0.682
first_marathon           

## Save

In [7]:
sample.to_csv("data/mcm_sample.csv", index=False)
sample.shape

(17900, 27)